In [2]:
import pandas as pd

# Ab pd.read_csv istemal karein
df_2019 = pd.read_csv('Divvy_Trips_2019_Q1.csv', encoding='latin1')
df_2020 = pd.read_csv('Divvy_Trips_2020_Q1.csv', encoding='latin1')

# Columns match karna
df_2019 = df_2019.rename(columns={
    'trip_id': 'ride_id',
    'start_time': 'started_at',
    'end_time': 'ended_at',
    'usertype': 'member_casual'
})

# Combine karna
all_trips = pd.concat([df_2019, df_2020], ignore_index=True)

# Dates aur calculations
all_trips['started_at'] = pd.to_datetime(all_trips['started_at'])
all_trips['ended_at'] = pd.to_datetime(all_trips['ended_at'])

all_trips['ride_length'] = (all_trips['ended_at'] - all_trips['started_at']).dt.total_seconds()
all_trips['day_of_week'] = all_trips['started_at'].dt.dayofweek

# Cleaning
all_trips_clean = all_trips[all_trips['ride_length'] > 0].copy()

print("Fauran ho gaya! Clean data ki rows:", len(all_trips_clean))

Fauran ho gaya! Clean data ki rows: 791746


In [3]:
# 1. Overall descriptive analysis for ride_length (in seconds / minutes)
print("--- Ride Length Summary (in seconds) ---")
print(all_trips_clean['ride_length'].describe())

# 2. Member vs Casual comparison (Mean ride length)
print("\n--- Average Ride Length by User Type ---")
print(all_trips_clean.groupby('member_casual')['ride_length'].mean())

# 3. Average ride length by day of the week and user type
print("\n--- Average Ride Length by Day & User Type ---")
print(all_trips_clean.groupby(['member_casual', 'day_of_week'])['ride_length'].mean())

# 4. Total number of rides by user type and day of the week
print("\n--- Total Rides by Day & User Type ---")
print(all_trips_clean.groupby(['member_casual', 'day_of_week']).size())

--- Ride Length Summary (in seconds) ---
count    7.917460e+05
mean     1.184130e+03
std      3.321714e+04
min      1.000000e+00
25%      3.280000e+02
50%      5.370000e+02
75%      9.100000e+02
max      1.063202e+07
Name: ride_length, dtype: float64

--- Average Ride Length by User Type ---
member_casual
Customer      3716.515607
Subscriber     833.572192
casual        5772.004620
member         760.626751
Name: ride_length, dtype: float64

--- Average Ride Length by Day & User Type ---
member_casual  day_of_week
Customer       0              2667.367865
               1              2427.289223
               2              3117.434311
               3              8027.532877
               4              3593.568000
               5              3619.791090
               6              2494.943176
Subscriber     0               878.075494
               1               862.020145
               2               725.719879
               3               720.587906
               4  

In [4]:
import pandas as pd

# 1. Cleaner labels ke liye member_casual column ko standardize karna (agar mixed hain)
all_trips_clean['user_type'] = all_trips_clean['member_casual'].replace({
    'Subscriber': 'Member',
    'Customer': 'Casual'
})

# 2. Table 1: Average Ride Length (Minutes mein convert karke taake parhne mein asani ho)
avg_length = all_trips_clean.groupby('user_type')['ride_length'].mean() / 60
table_avg_length = pd.DataFrame({
    'User Type': avg_length.index,
    'Average Ride Length (Minutes)Key': avg_length.values.round(2)
}).reset_index(drop=True)

print("📊 TABLE 1: Average Ride Length Comparison")
display(table_avg_length)

# 3. Table 2: Day of Week ke hisab se Total Rides aur Average Length
# (Note: 0 = Monday, 6 = Sunday)
day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
all_trips_clean['day_name'] = all_trips_clean['day_of_week'].map(day_names)

table_summary = all_trips_clean.groupby(['user_type', 'day_name']).agg(
    Total_Rides=('ride_id', 'count'),
    Avg_Ride_Minutes=('ride_length', lambda x: round(x.mean() / 60, 2))
).reset_index()

print("\n📊 TABLE 2: Day-by-Day Usage Summary (Rides & Duration)")
display(table_summary)

📊 TABLE 1: Average Ride Length Comparison


,User Type,Average Ride Length (Minutes)Key
0,Casual,61.94
1,Member,13.89
2,casual,96.20
3,member,12.68



📊 TABLE 2: Day-by-Day Usage Summary (Rides & Duration)


,user_type,day_name,Total_Rides,Avg_Ride_Minutes
0,Casual,Friday,3375,59.89
1,Casual,Monday,1892,44.46
2,Casual,Saturday,5993,60.33
3,Casual,Sunday,3766,41.58
4,Casual,Thursday,2920,133.79
5,Casual,Tuesday,2728,40.45
6,Casual,Wednesday,2489,51.96
7,Member,Friday,59672,13.89
8,Member,Monday,48507,14.63
9,Member,Saturday,29309,16.99


In [5]:
# Clean data ko CSV file ke tor par save karna
all_trips_clean.to_csv('Cyclistic_Cleaned_Trips.csv', index=False)

print("Aapki file 'Cyclistic_Cleaned_Trips.csv' ke naam se save ho chuki hai!")

Aapki file 'Cyclistic_Cleaned_Trips.csv' ke naam se save ho chuki hai!
